# text-only vs text-image：Hidden-state + Image-token Gradient 实验分析

本Notebook新增两条关键能力：
1. **按 layer_depth 的 hidden-state 空间分析**（PCA / t-SNE / 聚类）
2. **text-image 中 image token 梯度专项分析**（`grad_partition=image_only`）

用于在“整体结果相似”时挖掘更细粒度差异信号。


In [ ]:
import ast, json
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style='whitegrid', context='talk')


In [ ]:
GRAD_LOG_PATH = Path('gradient_logs.jsonl')
HS_LOG_PATH = Path('hidden_state_logs.jsonl')
OUT = Path('analysis_outputs')
OUT.mkdir(parents=True, exist_ok=True)

STEP_MIN=None
STEP_MAX=None
STEP_MOD=None
MAX_ROWS=250000
MAX_VEC_ROWS=80000
MAX_HS_ROWS=120000
MAX_TSNE_SAMPLES=4000

assert GRAD_LOG_PATH.exists(), GRAD_LOG_PATH
print('grad log:', GRAD_LOG_PATH.resolve())
print('hidden log exists:', HS_LOG_PATH.exists())


In [ ]:
def _step_ok(v, lo=None, hi=None, mod=None):
    try: s=int(v)
    except: return False
    if lo is not None and s<lo: return False
    if hi is not None and s>hi: return False
    if mod is not None and mod>1 and s%mod!=0: return False
    return True

def load_jsonl(path, usecols=None, row_filter=None, max_rows=None):
    usecols=set(usecols) if usecols else None
    rows=[]
    if not path.exists():
        return pd.DataFrame()
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line=line.strip()
            if not line: continue
            try: o=json.loads(line)
            except: continue
            if 'step' in o and not _step_ok(o['step'], STEP_MIN, STEP_MAX, STEP_MOD):
                continue
            if row_filter and not row_filter(o):
                continue
            rows.append(o if usecols is None else {k:o.get(k,None) for k in usecols})
            if max_rows and len(rows)>=max_rows: break
    return pd.DataFrame(rows)


## 1) Image-token梯度专项（text-image 核心）

In [ ]:
gdf = load_jsonl(
    GRAD_LOG_PATH,
    usecols=['step','layer_depth','adapter_type','param_name','grad_partition','grad_norm','grad_norm_per_token','partition_token_ratio','text_token_ratio','image_token_ratio','grad_path','grad_was_none'],
    max_rows=MAX_ROWS,
)
if len(gdf)==0:
    raise ValueError('No gradient rows loaded')

for c in ['step','layer_depth','grad_norm','grad_norm_per_token','partition_token_ratio','text_token_ratio','image_token_ratio']:
    if c in gdf.columns:
        gdf[c]=pd.to_numeric(gdf[c], errors='coerce')
if 'grad_was_none' in gdf.columns:
    gdf['grad_was_none']=gdf['grad_was_none'].fillna(False).astype(bool)

img_grad = gdf[gdf['grad_partition']=='image_only'].copy()
text_grad = gdf[gdf['grad_partition']=='text_only'].copy()
print('image_only rows:', len(img_grad), 'text_only rows:', len(text_grad))


In [ ]:
if len(img_grad)>0:
    # layer-depth聚合
    layer_img = img_grad.groupby('layer_depth', as_index=False).agg(
        mean_grad_per_token=('grad_norm_per_token','mean'),
        std_grad_per_token=('grad_norm_per_token','std'),
        mean_partition_ratio=('partition_token_ratio','mean'),
        none_ratio=('grad_was_none','mean'),
    )
    display(layer_img.head(12))
    layer_img.to_csv(OUT/'table_layer_image_token_grad_stats.csv', index=False, encoding='utf-8-sig')

    plt.figure(figsize=(11,5))
    sns.lineplot(data=layer_img, x='layer_depth', y='mean_grad_per_token', marker='o')
    plt.title('Image-token grad_norm_per_token by layer_depth')
    plt.tight_layout(); plt.savefig(OUT/'fig_layer_image_grad_per_token.png', dpi=180); plt.show()

# 与text token对比 gap
if len(img_grad)>0 and len(text_grad)>0:
    key=['step','layer_depth','adapter_type','param_name']
    wi=img_grad[key+['grad_norm_per_token']].rename(columns={'grad_norm_per_token':'img_g'})
    wt=text_grad[key+['grad_norm_per_token']].rename(columns={'grad_norm_per_token':'txt_g'})
    w=wi.merge(wt,on=key,how='inner')
    eps=1e-12
    w['rel_gap']=(w['img_g']-w['txt_g'])/(w['img_g']+w['txt_g']+eps)
    lg=w.groupby('layer_depth',as_index=False)['rel_gap'].agg(['mean','std','count']).reset_index()
    lg.columns=['layer_depth','rel_gap_mean','rel_gap_std','n']
    lg.to_csv(OUT/'table_layer_image_text_rel_gap.csv',index=False,encoding='utf-8-sig')

    plt.figure(figsize=(11,5))
    sns.lineplot(data=lg,x='layer_depth',y='rel_gap_mean',marker='o')
    plt.axhline(0,color='r',ls='--')
    plt.title('Layer-wise relative gap: image_only vs text_only')
    plt.tight_layout(); plt.savefig(OUT/'fig_layer_image_text_rel_gap.png', dpi=180); plt.show()


## 2) Hidden-state收集评估与按层空间分析（PCA / t-SNE / 聚类）

In [ ]:
hs_df = load_jsonl(
    HS_LOG_PATH,
    usecols=['step','layer_depth','modality','token_count','hidden_dim','hidden_path','hidden_norm_mean','hidden_norm_std'],
    max_rows=MAX_HS_ROWS,
)
if len(hs_df)==0:
    print('No hidden_state_logs.jsonl found or empty. Enable hidden-state logging in trainer args first.')
else:
    for c in ['step','layer_depth','token_count','hidden_dim','hidden_norm_mean','hidden_norm_std']:
        if c in hs_df.columns:
            hs_df[c]=pd.to_numeric(hs_df[c], errors='coerce')
    print('hidden-state rows:', len(hs_df), 'layers:', hs_df['layer_depth'].nunique())
    display(hs_df.head(5))


In [ ]:
def load_hidden_vectors(path_str):
    p=Path(path_str)
    if not p.exists():
        return None
    arr=np.load(p)
    if arr.ndim==1:
        arr=arr[None,:]
    return arr

if len(hs_df)>0:
    # 按layer-depth聚合基本统计
    hs_layer = hs_df.groupby(['layer_depth','modality'], as_index=False).agg(
        mean_token_count=('token_count','mean'),
        mean_norm=('hidden_norm_mean','mean'),
        std_norm=('hidden_norm_mean','std'),
    )
    hs_layer.to_csv(OUT/'table_layer_hidden_summary.csv', index=False, encoding='utf-8-sig')
    display(hs_layer.head(10))

    plt.figure(figsize=(11,5))
    sns.lineplot(data=hs_layer, x='layer_depth', y='mean_norm', hue='modality', marker='o')
    plt.title('Hidden norm by layer_depth and modality')
    plt.tight_layout(); plt.savefig(OUT/'fig_layer_hidden_norm_by_modality.png', dpi=180); plt.show()


In [ ]:
# 选择若干关键层做空间分布：PCA / t-SNE / 聚类
if len(hs_df)>0:
    candidate_layers = sorted(hs_df['layer_depth'].dropna().unique().tolist())
    # 取前中后层各1个（若存在）
    selected_layers = []
    if len(candidate_layers) > 0:
        selected_layers.append(candidate_layers[0])
    if len(candidate_layers) > 2:
        selected_layers.append(candidate_layers[len(candidate_layers)//2])
    if len(candidate_layers) > 1:
        selected_layers.append(candidate_layers[-1])
    selected_layers = sorted(list(set(selected_layers)))
    print('selected layers:', selected_layers)

    for layer in selected_layers:
        sdf = hs_df[hs_df['layer_depth']==layer].copy()
        parts = []
        labels = []
        for _, r in sdf.iterrows():
            arr = load_hidden_vectors(r.get('hidden_path', None))
            if arr is None:
                continue
            parts.append(arr)
            labels.extend([r['modality']] * arr.shape[0])

        if len(parts)==0:
            continue
        X = np.concatenate(parts, axis=0)
        y = np.array(labels)

        if X.shape[0] > MAX_TSNE_SAMPLES:
            idx = np.random.choice(X.shape[0], size=MAX_TSNE_SAMPLES, replace=False)
            X = X[idx]
            y = y[idx]

        # PCA
        pca = PCA(n_components=2, random_state=42)
        Zp = pca.fit_transform(X)
        pca_df = pd.DataFrame({'x':Zp[:,0], 'y':Zp[:,1], 'modality':y})

        plt.figure(figsize=(6,5))
        sns.scatterplot(data=pca_df, x='x', y='y', hue='modality', s=12, alpha=0.6)
        plt.title(f'Layer {layer} PCA hidden space')
        plt.tight_layout(); plt.savefig(OUT/f'fig_layer{int(layer)}_pca_hidden.png', dpi=180); plt.show()

        # t-SNE
        if X.shape[0] >= 50:
            tsne = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto')
            Zt = tsne.fit_transform(X)
            ts_df = pd.DataFrame({'x':Zt[:,0], 'y':Zt[:,1], 'modality':y})
            plt.figure(figsize=(6,5))
            sns.scatterplot(data=ts_df, x='x', y='y', hue='modality', s=12, alpha=0.6)
            plt.title(f'Layer {layer} t-SNE hidden space')
            plt.tight_layout(); plt.savefig(OUT/f'fig_layer{int(layer)}_tsne_hidden.png', dpi=180); plt.show()

        # 聚类 + silhouette
        n_clusters = min(4, max(2, len(np.unique(y))))
        km = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
        cluster = km.fit_predict(X)
        sil = silhouette_score(X, cluster) if X.shape[0] > n_clusters else np.nan

        # 模态-聚类纯度粗评估
        ct = pd.crosstab(pd.Series(y, name='modality'), pd.Series(cluster, name='cluster'))
        ct.to_csv(OUT/f'table_layer{int(layer)}_modality_cluster_crosstab.csv', encoding='utf-8-sig')

        print(f'Layer {layer}: silhouette={sil:.4f}, samples={X.shape[0]}, dim={X.shape[1]}')


## 3) 可执行优化建议（针对你当前“结果相似”的情况）

1. 先看 `table_layer_image_text_rel_gap.csv`，锁定高 gap 层再做局部机制验证。
2. 对这些层开启更高 `gradient_log_full_grad_max_params`，集中采样关键参数向量。
3. 开启 hidden-state logging 后，重点比较“前/中/后”层的 PCA/t-SNE 分离程度与 silhouette。
4. 若全局仍相似，优先报告“层局部差异 + 模块局部差异 + 空间分布差异”，避免只看全局均值曲线。